In [ ]:
import importlib
import warnings
from itertools import product
from pathlib import Path

import json
import pandas as pd
import semantic_engine_demo.search_metrics as sm
importlib.reload(sm)           # picks up edits to the module without restarting kernel

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT       = Path("/home/tommaso/code_base/semantic_engine_demo")
CORPUS_DIR = ROOT / "data" / "nomic_emb"
QUERY_DIR  = ROOT / "data" / "nomic_bench"
BENCH_FILE = ROOT / "data" / "benchmark" / "benchmark_queries_01.json"
OUT_DIR    = CORPUS_DIR

# ── What to test (edit freely) ─────────────────────────────────────────────
EXTRACTIONS = ["compact", "full", "structured"]
DIMS        = [768, 512, 256]   # ← add/remove dims here
TOP_K       = 10

# ── Build variant list ─────────────────────────────────────────────────────
VARIANTS = [
    sm.Variant(ext, dim, CORPUS_DIR, QUERY_DIR)
    for ext, dim in product(EXTRACTIONS, DIMS)
]
print(f"{len(VARIANTS)} variants configured")
for v in VARIANTS:
    c, q = v.exists()
    status = "✓" if (c and q) else f"✗ corpus={'✓' if c else '✗'} query={'✓' if q else '✗'}"
    print(f"  {status}  {v.label}")

In [ ]:
with open(BENCH_FILE) as f:
    BQ: list[dict] = json.load(f)
print(f"Loaded {len(BQ)} benchmark entries")

In [ ]:
all_results = sm.run_all_variants(VARIANTS, BQ, out_dir=OUT_DIR, k=TOP_K)

In [ ]:
summary = (
    pd.concat(all_results.values())
      .groupby(["variant", "dim", "extraction"])
      .agg(
          hit_rate  = ("hit",          "mean"),
          mrr       = ("mrr",          "mean"),
          ndcg      = ("ndcg",         "mean"),
          rec2      = ("recall_tier2", "mean"),
          n         = ("hit",          "count"),
      )
      .round(4)
      .sort_values("ndcg", ascending=False)
)
summary